# Lab 1 — RAG from scratch, end to end (EP10)

From the theory video: RAG gives the model a **library card** — index the library once, retrieve the relevant chunks, and let the model answer from them with citations (§16.1–16.2)

**Our library: this very textbook** — *The Hitchhiker's Guide to Agentic AI*, chapters 15–19 (pages 292–374), which came out after the model was trained

**What you'll do**
1. Ask the model about the textbook **without RAG**
2. **Indexing:** load the PDF with page numbers → split into chunks (512 tokens, overlap 64) → embed them
3. **Retrieve:** cosine similarity + top-k with numpy
4. **Answer:** the Listing 16.1 prompt, citing `[Doc N]` with page numbers
5. Three cases: found in the library · not in the library · **retrieval misses**

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.1–16.2 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep10-lab1-rag-from-scratch.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs about 0.1 US cent) |
| Packages | `openai`, `python-dotenv`, `pypdf`, `numpy`, `tokenizers` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"

## 1. Two models: one answers, one embeds

**The answering model `google/gemma-4-26b-a4b-it`** — the same model as every episode, pinned to Makora as in EP08–EP09.
USD 0.09 per million input tokens and USD 0.28 per million output tokens
· The `:free` version doesn't work (tested 2026-11-21: 3 out of 3 calls got 429 "temporarily rate-limited upstream")

**The embedding model `qwen/qwen3-embedding-8b`** — Qwen's open embedding model, called through OpenRouter with the same key.
It returns vectors of 4,096 numbers, at USD 0.01 per million tokens (embedding our whole library costs about 0.05 cents).
Pinned to the Nebius provider so the numbers are the same on every run

The `usage.cost` OpenRouter sends back is each request's real cost — we add it up in `COST` and look at it at the end

In [ ]:
import os, json, urllib.request
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def ask(prompt):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": prompt}])
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

## 2. Asking without RAG

The textbook came out in 2026, after Gemma was trained — does the model know this book?

In [ ]:
question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
print(ask(question))

## 3. Indexing step 1: load the documents with metadata

Download the textbook from arXiv, **version 1** (`…v1`) — newer versions add material and change the page numbers.
That's exactly why the book says RAG fits knowledge that changes: when the documents change, we just index again

We keep one record per page, each with metadata: the file name (`source`) and the page number (`page`) — and drop the running header repeated on every page

In [ ]:
URL = "https://arxiv.org/pdf/2606.24937v1"
urllib.request.urlretrieve(URL, "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
print(len(reader.pages), "pages")

HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = []
for n in range(292, 375):                      # book pages 292–374: chapters 15–19
    text = reader.pages[n - 1].extract_text().replace(HEADER, "")
    pages.append({"source": "hitchhikers-guide-v1.pdf", "page": n, "text": text})
print(len(pages), "pages loaded")
print(pages[3]["text"][:300])

## 4. Indexing step 2: split into chunks

Fixed size + overlap (§16.4.1): `W = 512` tokens per chunk, neighbouring chunks sharing `O = 64` tokens — counted with Gemma's tokenizer.
In this lab we chunk page by page, so every chunk has a single page number (Lab 3 tries other ways)

In [ ]:
tok = Tokenizer.from_pretrained(MODEL)

# ✏️ TODO 1 (▶ video 4:17): step from 0 by W − O tokens (stopping O tokens before the end) and cut W tokens per chunk

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    chunks = []
    for start in range(0, ___, ___):
        chunks.append({"source": page["source"], "page": page["page"],
                       "text": tok.decode(ids[___])})
    return chunks

chunks = [c for p in pages for c in chunk_page(p)]
print(len(chunks), "chunks")

## 5. Indexing step 3: embeddings

Send the texts to the embedding model 64 at a time, then **normalize** every vector to length 1.
Once they have length 1, cosine similarity (Equation 16.3) is just a dot product: `a @ b`

In [ ]:
def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64],
                                     extra_body=EMBED_EXTRA)
        COST["embed"] += r.usage.cost
        vectors += [d.embedding for d in r.data]
    # ✏️ TODO 2 (▶ video 5:17): make every row of V length 1 (divide by that row's norm)
    V = np.array(vectors)
    return ___

demo = embed(["car", "automobile", "banana"])
print(demo.shape)
print("car · automobile =", round(float(demo[0] @ demo[1]), 3))
print("car · banana     =", round(float(demo[0] @ demo[2]), 3))

In [ ]:
E = embed([c["text"] for c in chunks])
print(E.shape)

## 6. Retrieve: cosine similarity + top-k

Question → embedding (length 1) → `E @ q` scores every chunk at once → keep the `k` best-scoring chunks

In [ ]:
# ✏️ TODO 3 (▶ video 6:30): compute every chunk's cosine score, then pick the indices of the k highest
def search(query, k=3):
    q = embed([query])[0]
    scores = ___
    best = ___
    return [(float(scores[i]), chunks[i]) for i in best]

for score, c in search("What analogy does the textbook use to explain RAG?"):
    print(f"{score:.3f}  page {c['page']}  {c['text'][:70]!r}")

## 7. Answer: the Listing 16.1 prompt

Three rules: answer from the context **only** · if it isn't enough, **say so** · cite with `[Doc N]`.
Every chunk gets a `[Doc N] (Source: …, Page: …)` label — the metadata from part 3

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. Answer the question using ONLY
the provided context. If the context does not contain enough information,
say so explicitly. Cite your sources using [Doc N] notation."""

# ✏️ TODO 4 (▶ video 7:27): label every chunk [Doc N] (Source: …, Page: …), with N starting at 1

def build_rag_prompt(query, chunks):
    context_str = "\n\n".join(
        ___
        for i, c in enumerate(chunks))
    return f"""{SYSTEM_PROMPT}

Context:
{context_str}

Question: {query}

Answer:"""

# ✏️ TODO 5 (▶ video 7:49): retrieve k chunks, then ask the model with the prompt built from them

def rag(query, k=3):
    hits = ___
    for score, c in hits:
        print(f"  retrieved page {c['page']}  ({score:.3f})")
    return ___

print(rag("What analogy does the textbook use to explain RAG?"))

In [ ]:
print(rag("What chunk size and overlap does the textbook recommend for legal documents?"))

In [ ]:
print(rag("What is the capital of Australia?"))

## 8. When retrieval misses

Ask the lab's first question (part 2) through RAG — it's longer, and it has the book's title and the author's name in it

In [ ]:
print(rag(question))

In [ ]:
ranked = np.argsort(-(E @ embed([question])[0]))
where = [r for r, i in enumerate(ranked, 1) if "library card" in chunks[i]["text"]]
print("the 'library card' chunk is at rank", where)

In [ ]:
print(f"embeddings ${COST['embed']:.5f}  chat ${COST['chat']:.5f}")

## Summary

| Step | What we saw in the lab |
|---|---|
| No RAG | the model confidently said "open-book exam" — a phrase on no page of the book |
| Load + metadata | textbook v1 pages 292–374 → 83 pages, each with `source` + `page` |
| Chunk | 512 tokens, overlap 64 → 141 chunks |
| Embed | car · automobile ≈ 0.94 but car · banana ≈ 0.58 (not 0) — the order matters more than the number |
| Retrieve + generate | "library card", citing [Doc 2], page 295 · legal documents: paragraph level, 1-sentence overlap |
| Not in the library | "the context doesn't contain the information" — although the model knows the answer |
| Retrieval misses | the long question with the book's title → the "library card" chunk falls to rank 8 → "not in the context" |

**Exercises**
1. Run part 8 with `rag(question, k=8)` — does the model get it right, and how many times longer is the prompt?
2. Change `source` in part 3 to the book's full title and index again — does the model still complain that the context never mentions the book?

**Next — Lab 2:** write BM25 ourselves, compare it with dense retrieval on the same library, and merge the two with RRF